<a href="https://colab.research.google.com/github/cfcarmem/Especialista_IA_Alura_102026/blob/main/alura_analise_sentimento_pyton_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Vamos carregar o arquivo reviews.csv como um dataframe utilizando o pandas, que apelidamos de pd. Para isso, utilizamos a função pd.read_csv() para ler o arquivo CSV. Para obter o caminho completo do arquivo, copiamos o caminho da pasta e colamos entre aspas, pois é um texto. O caminho será //content//reviews.csv. Armazenamos isso em uma variável que podemos chamar de df_reviews, df_resenhas, ou df_avaliacoes, conforme preferirmos. em itálico*

In [ ]:
import pandas as pd

In [ ]:
#parte 1 - ler o csv com pandas

df_reviews = pd.read_csv('//content/reviews.csv')
df_reviews.head()


,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote
0,A2FII3I2MBMUIA,B007WTAJTO,1K3,it works as expected. I should have sprung for...,1356220800,23/12/2012,715,0,0
1,A3H99DFEG68SR,B007WTAJTO,1m2,This think has worked out great.Had a diff. br...,1384992000,21/11/2013,382,0,0
2,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",1373673600,13/07/2013,513,0,0
3,A2IDCSC6NVONIZ,B007WTAJTO,2Cents!,It's mini storage. It doesn't do anything els...,1367193600,29/04/2013,588,0,0
4,A26YHXZD5UFPVQ,B007WTAJTO,2K1Toaster,I have it in my phone and it never skips a bea...,1382140800,19/10/2013,415,0,0


In [ ]:
#passo2 - Fazer a análise de sentimento da coluna reviewText
somente_cinco_registros = df_reviews.head(3).copy()
coluna_de_reviews = somente_cinco_registros["reviewText"]
somente_cinco_registros

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote
0,A2FII3I2MBMUIA,B007WTAJTO,1K3,it works as expected. I should have sprung for...,1356220800,23/12/2012,715,0,0
1,A3H99DFEG68SR,B007WTAJTO,1m2,This think has worked out great.Had a diff. br...,1384992000,21/11/2013,382,0,0
2,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",1373673600,13/07/2013,513,0,0


In [ ]:
#passo 3 - vamos, para cada review, ver  análise de sentimento
#análise pode ser Positiva, Negativa ou Neutra
#Faremos com IA do Gemini

#importar gemini
import os  #biblioteca para guardar a chave em variável de ambiente no python
from google import genai #importando o gemini

from google.colab import userdata
os.environ["GOOGLE_API_KEY"]=userdata.get('gemini_api')

#vamos conectar o pytho no gemini

client = genai.Client()

In [ ]:
#vamos traduzir a coluna de reviews para lingua portuguesa
lista_coluna_traduzida = []
  for review_numero, resenha in enumerate(coluna_de_reviews):
    resposta = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"""Você irá traduzir a resenha que eu te mandarei abaixo para português.
        Segue a resenha a ser analisada: {resenha}
        """
    )
    lista_coluna_traduzida.append(resposta.text)
    print(f"Resenha {review_numero}: '{resenha}' -> traduzido é : {resposta.text}")
somente_cinco_registros["Opiniao"] = lista_coluna_traduzida

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,Opiniao
0,A2FII3I2MBMUIA,B007WTAJTO,1K3,it works as expected. I should have sprung for...,1356220800,23/12/2012,715,0,0,Funciona como o esperado. Eu deveria ter optad...
1,A3H99DFEG68SR,B007WTAJTO,1m2,This think has worked out great.Had a diff. br...,1384992000,21/11/2013,382,0,0,Essa coisa funcionou muito bem. Tinha um cartã...
2,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",1373673600,13/07/2013,513,0,0,"Comprei com embalagem de varejo, chegou origin..."


In [ ]:
print(somente_cinco_registros.columns.tolist())

['reviewerID', 'asin', 'reviewerName', 'reviewText', 'unixReviewTime', 'reviewTime', 'day_diff', 'helpful_yes', 'total_vote', 'Opiniao']


In [ ]:
print("Registros do DataFrame:", len(somente_cinco_registros))
print("Resenhas traduzidas:", len(lista_coluna_traduzida))

Registros do DataFrame: 3
Resenhas traduzidas: 3


In [ ]:
#passo4 Vamos chamar a LLM par analisar o sentimento
lista_de_analises_de_sentimentos = []
coluna_traduzida = somente_cinco_registros["Opiniao"]
for review_numero, resenha in enumerate(coluna_traduzida):
    resposta = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"""Você irá analisar a resenha que eu te mandarei abaixo, e retorna com uma análise de sentimento.
        Você deve responder APENAS com uma das seguintes palavras: 'Positiva', 'Negativa' ou 'Neutra',
        indicando o sentimento relativo aquela resenha específica.
        Exemplos:
        "Eu adorei esse produto" -> Positiva
        "Gostei, mas não é nada de especial" -> Neutra
        "Odiei esse produto" -> Negativa

        Segue a resenha a ser analisada: {resenha}
        """
    )
    lista_de_analises_de_sentimentos.append(resposta.text)
    print(f"Resenha {review_numero}: '{resenha}' -> Sentimento: {resposta.text}")


Resenha 0: 'Funciona como o esperado. Eu deveria ter optado pela capacidade maior. Acho que foi feito de forma um pouco mais barata do que as versões anteriores; a pintura não parece tão limpa quanto antes.' -> Sentimento: Neutra
Resenha 1: 'Essa coisa funcionou muito bem. Tinha um cartão de 64gb de outra marca e ele pifou depois de 3 meses. Este aqui tem aguentado muito bem desde que eu tinha meu S3, agora no meu Note3.*** atualização 21/03/14 Estou com ele há alguns meses e tive ZERO problemas desde que foi transferido do meu S3 para o meu Note3 e para um Note2. Este cartão é confiável e sólido! Valeu!' -> Sentimento: Positiva


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 13h22m14.680174635s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See ht

In [ ]:
while len(lista_de_analises_de_sentimentos) < len(somente_cinco_registros):
    lista_de_analises_de_sentimentos.append("Pendente")

somente_cinco_registros["Sentimento"] = lista_de_analises_de_sentimentos

In [ ]:
#passo5 - Vamos criar uma nova coluna no dataFrame para armazenar a análise

#o tamanho da lista com a análise tem que ser o mesmo tamanho do dataFrame, senão dá erro.

#somente_cinco_registros["Análise de Sentimento"] = lista_de_analises_de_sentimentos

ValueError: Length of values (2) does not match length of index (3)

In [ ]:
somente_cinco_registros

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,Opiniao,Sentimento
0,A2FII3I2MBMUIA,B007WTAJTO,1K3,it works as expected. I should have sprung for...,1356220800,23/12/2012,715,0,0,Funciona como o esperado. Eu deveria ter optad...,Neutra
1,A3H99DFEG68SR,B007WTAJTO,1m2,This think has worked out great.Had a diff. br...,1384992000,21/11/2013,382,0,0,Essa coisa funcionou muito bem. Tinha um cartã...,Positiva
2,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",1373673600,13/07/2013,513,0,0,"Comprei com embalagem de varejo, chegou origin...",Pendente
